# Cytnx PR #1230 — GPU Float64 `Tensor::get` L2 benchmark

Baseline: PR #1230, SHA `8de452fa550e0fa8fe3a4d0c2b8d28afa037bffe`. Experimental branch: `bench/1230-gpu-get-optimization`.

**Kaggle settings:** Accelerator = NVIDIA GPU, Internet = ON, notebook private. This notebook uses a native CUDA **Release** build (no coverage, ASan or Python bindings), correctness-checks every S0–S4 case, executes a stratified pilot and full Google Benchmark suite, then writes JSON, CSV and Markdown results.

S0 is the original Double backend; S1 flat, S2 warp, S3 tiled, S4 pilot-trained policy. Re-run the cells in sequence. A Kaggle session may have two GPUs; `CUDA_VISIBLE_DEVICES=0` deliberately selects one.


In [ ]:
import os, subprocess, pathlib
def run(*args, cwd=None):
    print('+', ' '.join(map(str,args)), flush=True)
    subprocess.run(args, cwd=cwd, check=True)
run('nvidia-smi')
run('bash','-lc','command -v nvcc && nvcc --version || true')
print('Disk:')
run('df','-h','/kaggle/working')


In [ ]:
import pathlib, subprocess
ROOT = pathlib.Path('/kaggle/working/cytnx')
BRANCH = 'bench/1230-gpu-get-optimization'
if not (ROOT / '.git').exists():
    subprocess.run(['git','clone','--filter=blob:none','--branch',BRANCH,'https://github.com/Cytnx-dev/Cytnx.git',str(ROOT)],check=True)
else:
    subprocess.run(['git','fetch','origin',BRANCH],cwd=ROOT,check=True)
    subprocess.run(['git','checkout',BRANCH],cwd=ROOT,check=True)
    subprocess.run(['git','pull','--ff-only'],cwd=ROOT,check=True)
subprocess.run(['git','submodule','update','--init','--recursive'],cwd=ROOT,check=True)
SHA = subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip()
print('Benchmark branch SHA:', SHA)


In [ ]:
import subprocess, os
env = os.environ.copy()
env['CYTNX_BUILD_JOBS'] = '2'
env['CYTNX_GET_OUTPUT_DIR'] = '/kaggle/working/cytnx/results/gpu_get'
subprocess.run(['bash','benchmarks/gpu/kaggle_run.sh'],cwd=ROOT,env=env,check=True)


In [ ]:
import pathlib, shutil
from IPython.display import display, Markdown
results = ROOT / 'results' / 'gpu_get'
report = results / 'gpu_get_report.md'
if report.exists():
    display(Markdown(report.read_text()))
    archive = shutil.make_archive('/kaggle/working/cytnx_gpu_get_bench_results', 'zip', root_dir=results)
    print('Results archive:', archive)
else:
    print('Full benchmark results unavailable. Inspect',results/'kaggle_run.log')
    if (results/'kaggle_run.log').exists():
        print((results/'kaggle_run.log').read_text()[-4000:])
